In [32]:
import sys
sys.path.insert(0, "..")  # project root, so `src` is importable from notebooks/

from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

from src.data.prepare import prepare_splits
from src.data.caption_cleaning import clean_caption

## 1. Load, clean and split

Load the two datasets, then `prepare_splits` cleans the `description` column into a `captions` column and carves a stratified validation set (15%, seed 42) out of train. Test is never split.

In [33]:
train = load_dataset("Multimodal-Fatima/CUB_train", split="train")
test = load_dataset("Multimodal-Fatima/CUB_test", split="test")

data = prepare_splits(train, test, val_fraction=0.15, seed=42)

print(data)
print("columns:", data["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['image', 'description', 'label', 'file_name', 'id', 'captions'],
        num_rows: 5094
    })
    validation: Dataset({
        features: ['image', 'description', 'label', 'file_name', 'id', 'captions'],
        num_rows: 900
    })
    test: Dataset({
        features: ['image', 'description', 'label', 'file_name', 'id', 'captions'],
        num_rows: 5794
    })
})
columns: ['image', 'description', 'label', 'file_name', 'id', 'captions']


## 2. Cleaning: before and after

Reading `ds["column"]` only touches that column, so no images are decoded.

In [34]:
raw = {name: ds["description"] for name, ds in data.items()}
caps = {name: ds["captions"] for name, ds in data.items()}

i = 1
print("RAW:")
print(repr(raw["train"][i]))
print("\nCLEANED:")
for c in caps["train"][i]:
    print(" -", c)

RAW:
"this bird has a white body with grey wings, a yellow beak which curves down, and webbed feet.\na white head, neck, breast and belly bird with grey and white wings and black and white short tail, yellow bill, pink tarsus and feet.\na wading bird with an orange beak and webbed feet, its head and breast are white with black wings\nbird with light brown beak, white crown, nape, throat, breast, belly and abdomen, and gray wings\nthis bird is white with grey on its wings and has a long, pointy beak.\nthe bird is medium to large size with a long yellow bill, white crown and breast, gray wings, and pinkish tan webbed feet.\nthis bird has wings that are grey and has a white body\nthis bird has a white crown with a yellow bill and grey coverts.\nthis bird has a short broad beak and a small head compared the rest of it's body and covered in grey, white, and black feathers.\nthis bird has a white crown as well as a orange bill\n"

CLEANED:
 - this bird has a white body with grey wings a yell

In [35]:
def summary(raw_descs, cleaned):
    lines = sum(len(d.split("\n")) for d in raw_descs)
    nonempty = sum(1 for d in raw_descs for l in d.split("\n") if clean_caption(l))
    kept = sum(len(c) for c in cleaned)
    per_image = pd.Series([len(c) for c in cleaned])
    return {
        "images": len(cleaned), "lines after split": lines,
        "empty dropped": lines - nonempty, "duplicates dropped": nonempty - kept,
        "captions kept": kept, "min per image": per_image.min(), "max per image": per_image.max(),
    }

pd.DataFrame({name: summary(raw[name], caps[name]) for name in data}).T

,images,lines after split,empty dropped,duplicates dropped,captions kept,min per image,max per image
train,5094,56034,5094,556,50384,7,10
validation,900,9900,900,95,8905,8,10
test,5794,63734,5794,684,57256,7,10


In [27]:
# sanity checks: no image without captions, no caption empty or with odd characters
for name, c_list in caps.items():
    assert all(len(c) > 0 for c in c_list), f"{name}: image with no captions"
    assert all(cap and cap == cap.strip() and "  " not in cap for c in c_list for cap in c), f"{name}: bad whitespace"
    assert all(all(ch.isalpha() or ch == " " for ch in cap) for c in c_list for cap in c), f"{name}: unexpected characters"
print("all checks passed")

all checks passed


## 3. Split sizes and stratification

In [36]:
sizes = pd.Series({name: len(ds) for name, ds in data.items()})
print(sizes.to_string())
print(f"validation share of the original train set: {sizes['validation'] / (sizes['train'] + sizes['validation']):.1%}")

# every class should appear in train and validation, with 4 or 5 validation images each
tr = Counter(data["train"]["label"]); va = Counter(data["validation"]["label"])
print("classes in train:", len(tr), "| classes in val:", len(va))
print("val images per class (min / max):", min(va.values()), "/", max(va.values()))

train         5094
validation     900
test          5794
validation share of the original train set: 15.0%
classes in train: 200 | classes in val: 200
val images per class (min / max): 4 / 5
